[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLAlchemy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)

# Many to Many


## What you will be able to do

Link two classes whose rows pair up many to many through a table of pairs, with `secondary`, and
add, remove and query the pairs from either side. Say when the table of pairs holds data of its own,
as an enrollment holds a grade, and map it as an association class instead, and keep a read-only
`secondary` beside it. Recognize the mapped class passed where a table belongs, a pair added twice,
two mappings that fight over one table, and the grade that a `secondary` relationship cannot reach.


## The idea

### The problem

The college's courses satisfy general education requirements. Statistics counts as Quantitative
Reasoning and as Social Science, and Quantitative Reasoning is met by four different courses. Neither
table can hold that with a foreign key: a course would need a column for every requirement it meets,
and a requirement one for every course. The pairs go in a table of their own, `course_requirements`,
with a course's id and a requirement's id in each row, and the registrar's code wants
`statistics.requirements` and `quantitative.courses` as lists, with the table of pairs kept out of
sight.

Students and sections are paired the same way, through `enrollments`, with one difference: an
enrollment is not just a pair. It has a status and a grade, which belong to neither the student nor
the section but to the two together. A relationship that hides the table of pairs hides those columns
too, so a transcript cannot be built from it, and a grade cannot be recorded through it.

### What a many to many relationship is

> A **many to many** relationship links two classes through an **association table**, a table whose
> rows are pairs of foreign keys, one to each side. **`relationship(secondary=table)`** maps it:
> every course's `requirements` is a list of `Requirement` objects, every requirement's `courses` a
> list of `Course` objects, and appending to either list inserts a pair, removing deletes one. The
> association table is a `Table`, never a mapped class, because `secondary` manages its rows by
> itself. When the pairs carry data of their own, the table is mapped as an **association object**
> instead, a class with a relationship to each side, as `Enrollment` already is, and a `secondary`
> relationship over it is marked `viewonly=True`, for reading only.

### Why it works that way

- **The pairs are a table, not a class.** `secondary` inserts and deletes rows of the association
  table itself, so it needs the table and would conflict with a class that did the same.
- **Both sides are lists.** `course.requirements.append(requirement)` also puts the course in
  `requirement.courses` when the two relationships name each other with `back_populates`, exactly as
  for one to many.
- **A pair is a row, and a row is unique.** The association table's primary key is the pair, so the
  same pair twice is refused by the database, although a Python list would happily hold it twice.
- **`secondary` knows two columns.** It writes and joins on the two foreign keys and nothing else, so
  any other column of the table is out of its reach: never read, and written only by the table's own
  default.
- **Data on the pair means a class for the pair.** An association object has relationships to both
  sides, so a student reaches her sections through her enrollments, and every enrollment's grade is
  on the way.

### Where this shows up

Tags on a blog post, members of a group, the products in an order, the roles of a user: most
applications have several of these. SQLModel, in the **SQLModel, Deep Dive** guide, spells the same
two patterns with `link_model`. The **Loading Strategies** notebook loads both kinds of collection
without a query for every row, and the **Joins** notebook of the **sqlite3, Deep Dive** guide wrote
the join through a table of pairs by hand.

### What this notebook covers

- An association table, and `secondary` on both sides
- Linking and unlinking, from either side
- Querying through a many to many relationship
- When the pair has data of its own: the association object, and `viewonly`
- Which of the two mappings to use when
- A degree audit, finished
- Four errors, from a class passed as `secondary` to the grade that `secondary` cannot reach

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
from sqlalchemy import Column, ForeignKey, Table, create_engine, func, select
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column, relationship


class Base(DeclarativeBase):
    pass


links = Table("course_requirements", Base.metadata,
              Column("course_id", ForeignKey("courses.id"), primary_key=True),
              Column("requirement_id", ForeignKey("requirements.id"), primary_key=True))


class Course(Base):
    __tablename__ = "courses"
    id: Mapped[int] = mapped_column(primary_key=True)
    code: Mapped[str]
    requirements: Mapped[list["Requirement"]] = relationship(secondary=links,
                                                             back_populates="courses")


class Requirement(Base):
    __tablename__ = "requirements"
    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str]
    courses: Mapped[list["Course"]] = relationship(secondary=links, back_populates="requirements")


engine = create_engine("sqlite://")
Base.metadata.create_all(engine)
with Session(engine) as session:
    statistics = Course(code="STA-200")
    statistics.requirements = [Requirement(name="Quantitative"), Requirement(name="Social Science")]
    session.add(statistics)
    session.commit()
    print([requirement.name for requirement in statistics.requirements])
    print(session.scalar(select(func.count()).select_from(links)), "rows in the association table")
```

```
['Quantitative', 'Social Science']
2 rows in the association table
```

One course with two requirements, and neither class mentions the other's id: the association table
holds the pairs, and `secondary` inserted a row into it for each.


## Setup

Eleven imports, and the college built from its classes.

- `sqlalchemy` is the library itself, and the cell prints its version
- `Table` and `Column`, from `sqlalchemy`, describe the association table, and `relationship`, from
  `sqlalchemy.orm`, maps both kinds of many to many, with the rest of what the classes need
- `select`, `func`, `insert`, `inspect` and `text` read the tables, and `create_engine` and `event`
  make the engine
- `IntegrityError`, from `sqlalchemy.exc`, is the error a pair added twice raises
- `warnings` catches the warning that one of the Common errors produces
- `StaticPool`, from `sqlalchemy.pool`, is the pool the helper uses for a database in memory
- `date` is what the `Date` columns take and return
- `logging` carries the SQL an engine logs to `PrintStatements`
- `Path` names the files, and `shutil` removes the scratch folder at the start and at the end

From this notebook on, Setup builds the college from the classes of the **Relationships** notebook,
with a relationship on each side of every foreign key.

Colab has SQLAlchemy installed, and this notebook runs version 2.0.54. Any 2.0 release runs it,
though an error may be worded a little differently. To match it exactly, run
`%pip install sqlalchemy==2.0.54` in a cell of its own, restart the session, and run this cell again.


In [1]:
import logging
import shutil
import warnings
from datetime import date
from pathlib import Path

import sqlalchemy
from sqlalchemy import (CheckConstraint, Column, ForeignKey, MetaData, String, Table, UniqueConstraint, create_engine,
                        event, func, insert, inspect, select, text)
from sqlalchemy.exc import IntegrityError
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column, relationship, sessionmaker
from sqlalchemy.pool import StaticPool

SCRATCH = Path("scratch")
shutil.rmtree(SCRATCH, ignore_errors=True)
SCRATCH.mkdir()
DATABASE = SCRATCH / "college.db"

NAMES = [
    "Ana Reyes", "Ben Okafor", "Chloe Martin", "Daniel Kim", "Elena Petrova", "Felix Wagner",
    "Grace Lin", "Hassan Ali", "Isabel Costa", "Jonas Berg", "Keiko Tanaka", "Liam Murphy",
    "Maya Patel", "Noah Andersen", "Olivia Brandt", "Pavel Novak", "Quinn Harper", "Rosa Delgado",
    "Sam Ito", "Tara Nilsen", "Umar Farouk", "Vera Kowalski", "Wes Carter", "Yara Haddad",
    "Aoife O'Brien",
]
PROGRAMS = ["Biology", "Computer Science", "Mathematics", "Psychology", "History"]
TERMS = [("Fall 2024", "2024-08-26"), ("Spring 2025", "2025-01-13"), ("Fall 2025", "2025-08-25"),
         ("Spring 2026", "2026-01-12")]
STUDENTS = [(name, f"{name[0]}{name.split()[-1]}@college.edu".lower().replace("'", ""),
             PROGRAMS[i % len(PROGRAMS)], TERMS[i % 3][1]) for i, name in enumerate(NAMES)]
COURSES = [
    ("BIO-101", "Introduction to Biology", "Biology", 4),
    ("CHE-110", "General Chemistry", "Chemistry", 4),
    ("MAT-120", "Calculus I", "Mathematics", 4),
    ("MAT-121", "Calculus II", "Mathematics", 4),
    ("CSC-101", "Programming I", "Computer Science", 3),
    ("CSC-201", "Data Structures", "Computer Science", 3),
    ("ENG-105", "Composition", "English", 3),
    ("HIS-110", "World History", "History", 3),
    ("PSY-101", "Introduction to Psychology", "Psychology", 3),
    ("STA-200", "Statistics", "Mathematics", 3),
]
GRADES = ["A", "A-", "B+", "B", "B-", "C+", "C", "C-", "D", "F"]

# One section of every course in every term, so the section of course c in term t has id (t - 1) * 10 + c.
SECTIONS = [(course, term, 30) for term in range(1, len(TERMS) + 1) for course in range(1, len(COURSES) + 1)]

# Three courses a term for every student, from the term they started. Spring 2026 is under way.
ENROLLMENTS = []
for s in range(len(NAMES)):
    for term in range(s % 3 + 1, len(TERMS) + 1):
        for k in range(3):
            section = (term - 1) * len(COURSES) + (s + term + 3 * k) % len(COURSES) + 1
            if term < len(TERMS):
                ENROLLMENTS.append((s + 1, section, "completed", GRADES[(s * 7 + term * 5 + k * 3) % len(GRADES)]))
            else:
                ENROLLMENTS.append((s + 1, section, "enrolled", None))

class PrintStatements(logging.Handler):
    """Print what an engine logs, leaving out the time: every statement, and the values sent with it."""

    def emit(self, record):
        if record.msg == "[%s] %r":                  # after a statement: how long it took, then its values
            values = repr(record.args[1])
            if values != "()":
                print("    values:", values)
        else:
            for line in record.getMessage().splitlines():
                print("   ", line.rstrip())


sql_log = logging.getLogger("sqlalchemy.engine.Engine")
sql_log.handlers = [PrintStatements()]              # this handler alone, however often the cell runs
sql_log.propagate = False                           # and no handler above it prints the same lines again


def college_engine(path=None, echo=False):
    """An engine for the college's database, in a file or in memory, with foreign keys enforced."""
    if path is None:                                # in memory: one connection, and one database, for every thread
        engine = create_engine("sqlite://", poolclass=StaticPool, echo=echo,
                               connect_args={"check_same_thread": False, "autocommit": False})
    else:
        engine = create_engine(f"sqlite:///{path}", echo=echo, connect_args={"autocommit": False})

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(dbapi_connection, connection_record):
        dbapi_connection.autocommit = True          # the PRAGMA does nothing inside a transaction,
        dbapi_connection.execute("PRAGMA foreign_keys = ON")
        dbapi_connection.autocommit = False         # and with autocommit=False sqlite3 keeps one open

    return engine

NAMING = {
    "pk": "pk_%(table_name)s",
    "uq": "uq_%(table_name)s_%(column_0_N_name)s",
    "ck": "ck_%(table_name)s_%(constraint_name)s",
    "fk": "fk_%(table_name)s_%(column_0_name)s_%(referred_table_name)s",
    "ix": "ix_%(column_0_label)s",
}


GRADE_POINTS = {"A": 4.0, "A-": 3.7, "B+": 3.3, "B": 3.0, "B-": 2.7, "C+": 2.3, "C": 2.0, "C-": 1.7, "D": 1.0, "F": 0.0}


class Base(DeclarativeBase):
    metadata = MetaData(naming_convention=NAMING)


class Student(Base):
    __tablename__ = "students"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))
    email: Mapped[str] = mapped_column(String(200), unique=True)
    program: Mapped[str] = mapped_column(String(50))
    started_on: Mapped[date]

    enrollments: Mapped[list["Enrollment"]] = relationship(back_populates="student", order_by="Enrollment.section_id")

    def __repr__(self):
        return f"Student({self.name!r}, {self.program!r})"


class Course(Base):
    __tablename__ = "courses"
    __table_args__ = (CheckConstraint("credits BETWEEN 1 AND 6", name="credits_range"),)

    id: Mapped[int] = mapped_column(primary_key=True)
    code: Mapped[str] = mapped_column(String(10), unique=True)
    title: Mapped[str] = mapped_column(String(100))
    department: Mapped[str] = mapped_column(String(50))
    credits: Mapped[int]

    sections: Mapped[list["Section"]] = relationship(back_populates="course", order_by="Section.term_id")

    def __repr__(self):
        return f"Course({self.code!r}, {self.credits})"


class Term(Base):
    __tablename__ = "terms"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(20), unique=True)
    starts_on: Mapped[date]

    sections: Mapped[list["Section"]] = relationship(back_populates="term", order_by="Section.course_id")

    def __repr__(self):
        return f"Term({self.name!r})"


class Section(Base):
    __tablename__ = "sections"
    __table_args__ = (UniqueConstraint("course_id", "term_id"), CheckConstraint("capacity > 0", name="capacity_positive"))

    id: Mapped[int] = mapped_column(primary_key=True)
    course_id: Mapped[int] = mapped_column(ForeignKey("courses.id"))
    term_id: Mapped[int] = mapped_column(ForeignKey("terms.id"))
    capacity: Mapped[int]

    course: Mapped["Course"] = relationship(back_populates="sections")
    term: Mapped["Term"] = relationship(back_populates="sections")
    enrollments: Mapped[list["Enrollment"]] = relationship(back_populates="section", order_by="Enrollment.student_id")

    def __repr__(self):
        return f"Section({self.id})"


class Enrollment(Base):
    __tablename__ = "enrollments"
    __table_args__ = (CheckConstraint("status IN ('enrolled', 'completed', 'withdrawn')", name="status_known"),)

    student_id: Mapped[int] = mapped_column(ForeignKey("students.id"), primary_key=True)
    section_id: Mapped[int] = mapped_column(ForeignKey("sections.id"), primary_key=True)
    status: Mapped[str] = mapped_column(String(20), server_default="enrolled")
    grade: Mapped[str | None] = mapped_column(String(2))

    student: Mapped["Student"] = relationship(back_populates="enrollments")
    section: Mapped["Section"] = relationship(back_populates="enrollments")

    @property
    def grade_points(self):
        """The points the grade is worth, or None before there is a grade."""
        return None if self.grade is None else GRADE_POINTS[self.grade]

    def __repr__(self):
        return f"Enrollment(student {self.student_id}, section {self.section_id}, {self.grade!r})"


def build_college(engine):
    """Create the college's tables from the classes, load the lists above into them, and count their rows."""
    Base.metadata.create_all(engine)
    rows = {
        Course: [{"code": code, "title": title, "department": department, "credits": credits}
                 for code, title, department, credits in COURSES],
        Student: [{"name": name, "email": email, "program": program, "started_on": date.fromisoformat(started)}
                  for name, email, program, started in STUDENTS],
        Term: [{"name": name, "starts_on": date.fromisoformat(starts)} for name, starts in TERMS],
        Section: [{"course_id": course, "term_id": term, "capacity": capacity} for course, term, capacity in SECTIONS],
        Enrollment: [{"student_id": student, "section_id": section, "status": status, "grade": grade}
                     for student, section, status, grade in ENROLLMENTS],
    }
    with engine.begin() as conn:
        for cls, values in rows.items():
            conn.execute(insert(cls), values)
        return {cls.__tablename__: conn.execute(select(func.count()).select_from(cls)).scalar_one() for cls in rows}


engine = college_engine(DATABASE)
print("sqlalchemy", sqlalchemy.__version__, "|", DATABASE, "|", build_college(engine))

SessionLocal = sessionmaker(engine)


sqlalchemy 2.0.54 | scratch/college.db | {'courses': 10, 'students': 25, 'terms': 4, 'sections': 40, 'enrollments': 228}


## Worked examples

### An association table, and secondary on both sides

The table of pairs is a `Table` in the classes' own `MetaData`, with a foreign key to each side and
the pair as its primary key. `Requirement` gets `courses`, and `Course`, already declared, gains
`requirements`: a mapped class accepts a new relationship assigned to it, which suits a notebook,
and in a program the attribute would sit in the class with the others:


In [2]:
course_requirements = Table(
    "course_requirements", Base.metadata,
    Column("course_id", ForeignKey("courses.id"), primary_key=True),
    Column("requirement_id", ForeignKey("requirements.id"), primary_key=True),
)


class Requirement(Base):
    __tablename__ = "requirements"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(50), unique=True)

    courses: Mapped[list["Course"]] = relationship(secondary=course_requirements, back_populates="requirements",
                                                   order_by="Course.code")

    def __repr__(self):
        return f"Requirement({self.name!r})"


Course.requirements = relationship(Requirement, secondary=course_requirements, back_populates="courses",
                                   order_by=Requirement.name)
Base.metadata.create_all(engine)                    # creates the two new tables, and leaves the rest alone

for rel in (Course.requirements, Requirement.courses):
    print(rel, rel.property.direction.name, "through", rel.property.secondary.name)
print("tables now:", sorted(inspect(engine).get_table_names()))


Course.requirements MANYTOMANY through course_requirements
Requirement.courses MANYTOMANY through course_requirements
tables now: ['course_requirements', 'courses', 'enrollments', 'requirements', 'sections', 'students', 'terms']


Both sides are `MANYTOMANY`, through `course_requirements`, and `create_all` added the two new tables
to the database and left the five it already had alone.

### Linking and unlinking, from either side

A requirement is made with its list of courses, and every course in the list gets a row in the
association table. `SATISFIES` says which courses meet which requirement:


In [3]:
SATISFIES = {
    "Humanities": ["HIS-110"],
    "Lab Science": ["BIO-101", "CHE-110"],
    "Quantitative Reasoning": ["CSC-101", "MAT-120", "MAT-121", "STA-200"],
    "Social Science": ["PSY-101", "STA-200"],
    "Writing": ["ENG-105", "HIS-110"],
}

with SessionLocal.begin() as session:
    courses = {course.code: course for course in session.scalars(select(Course))}
    for name, codes in SATISFIES.items():
        session.add(Requirement(name=name, courses=[courses[code] for code in codes]))

with SessionLocal() as session:
    statistics = session.scalars(select(Course).where(Course.code == "STA-200")).one()
    print("Statistics meets:          ", statistics.requirements)
    data_structures = session.scalars(select(Course).where(Course.code == "CSC-201")).one()
    print("Data Structures meets:     ", data_structures.requirements)
    writing = session.scalars(select(Requirement).where(Requirement.name == "Writing")).one()
    print("Writing is met by:         ", writing.courses)
    print("rows in course_requirements:", session.execute(text("SELECT COUNT(*) FROM course_requirements")).scalar_one())


Statistics meets:           [Requirement('Quantitative Reasoning'), Requirement('Social Science')]
Data Structures meets:      []
Writing is met by:          [Course('ENG-105', 3), Course('HIS-110', 3)]
rows in course_requirements: 11


Every pair was written once, from the requirement's side, and reads back from both: Statistics lists
its two requirements through `back_populates`, although no line of code ever appended to
`statistics.requirements`. Data Structures meets nothing, so its list is empty. Eleven rows, one for
every pair in `SATISFIES`. Removing a course from a requirement deletes the pair, and only the pair:


In [4]:
engine.echo = True
with SessionLocal() as session:                      # a trial run: nothing in this block is committed
    statistics = session.scalars(select(Course).where(Course.code == "STA-200")).one()
    social = session.scalars(select(Requirement).where(Requirement.name == "Social Science")).one()
    engine.echo = False
    print("before:", statistics.requirements)
    statistics.requirements.remove(social)
    engine.echo = True
    session.flush()
    engine.echo = False
    print("after: ", statistics.requirements)
    print("requirements in their table:", session.scalar(select(func.count()).select_from(Requirement)))
    session.rollback()


    BEGIN (implicit)
    SELECT courses.id, courses.code, courses.title, courses.department, courses.credits
    FROM courses
    WHERE courses.code = ?
    values: ('STA-200',)
    SELECT requirements.id, requirements.name
    FROM requirements
    WHERE requirements.name = ?
    values: ('Social Science',)
before: [Requirement('Quantitative Reasoning'), Requirement('Social Science')]
    DELETE FROM course_requirements WHERE course_requirements.course_id = ? AND course_requirements.requirement_id = ?
    values: (10, 4)
after:  [Requirement('Quantitative Reasoning')]
requirements in their table: 5


The flush sent one `DELETE` to `course_requirements`, and the course and the requirement both
survived, only no longer paired. Nothing was committed, so the rollback put the pair back. The
**Cascades and Deletes** notebook is about deleting the objects themselves.

### Querying through a many to many relationship

A many to many relationship is a join in a query too, through the association table, which the SQL
shows:


In [5]:
PER_REQUIREMENT = (
    select(Requirement.name, func.count(Course.id))
    .join(Requirement.courses)
    .group_by(Requirement.name)
    .order_by(Requirement.name)
)
print(" ".join(str(PER_REQUIREMENT.compile(engine)).split()))

with SessionLocal() as session:
    print(session.execute(PER_REQUIREMENT).all())


SELECT requirements.name, count(courses.id) AS count_1 FROM requirements JOIN course_requirements AS course_requirements_1 ON requirements.id = course_requirements_1.requirement_id JOIN courses ON courses.id = course_requirements_1.course_id GROUP BY requirements.name ORDER BY requirements.name
[('Humanities', 1), ('Lab Science', 2), ('Quantitative Reasoning', 4), ('Social Science', 2), ('Writing', 2)]


`.join(Requirement.courses)` became two joins, one to the association table and one on to `courses`,
written from the foreign keys. The counts agree with `SATISFIES`.

### When the pair has data of its own: the association object, and viewonly

`enrollments` pairs students with sections, and holds a status and a grade as well, so the college's
classes map it as a class, `Enrollment`, with a relationship to each side. A `secondary`
relationship over the same table is still handy for reading a student's sections directly, and
`viewonly=True` makes it read only, so that only `Enrollment` ever writes to the table:


In [6]:
Student.sections = relationship(Section, secondary="enrollments", viewonly=True, order_by=Section.id)

with SessionLocal() as session:
    chloe = session.get(Student, 3)
    print("through secondary:  ", chloe.sections)
    print("through Enrollment: ", [(enrollment.section, enrollment.grade) for enrollment in chloe.enrollments])


through secondary:   [Section(22), Section(26), Section(29), Section(33), Section(37), Section(40)]
through Enrollment:  [(Section(22), 'C+'), (Section(26), 'F'), (Section(29), 'B+'), (Section(33), None), (Section(37), None), (Section(40), None)]


The same six sections both ways. `chloe.sections` skips the enrollment and has no grade to give,
while `chloe.enrollments` passes through it, with the grade and the status on the way.
`secondary="enrollments"` names the table by its name, which the classes' `MetaData` resolves.

### Which of the two mappings to use when

| Use | When | Why |
|---|---|---|
| `secondary=` a `Table` of two foreign keys | a pair and nothing more, such as a course and a requirement | the pairs stay out of sight, and both sides are plain lists |
| an association object, a class with a relationship to each side | a pair with data of its own, such as an enrollment with a grade | the data is an attribute of the pair |
| `secondary=` with `viewonly=True` beside an association object | reading straight through to the other side | a shortcut that cannot write, so only the class writes |
| `Mapped[set[...]]` instead of `Mapped[list[...]]` | a collection that must never hold one object twice | a set refuses a second copy before the database is asked |

The default for a plain pair is `secondary` with `back_populates` on both sides, and a class the
moment the pair needs a column of its own.

### A degree audit, finished

The pieces of this notebook in one function. `audit` walks a student's enrollments, the association
object, to the courses she passed, and asks each course's `requirements`, the `secondary`
relationship, which requirements it meets. It reports every requirement with the first passed course
that meets it, or `None`:


In [7]:
def audit(session, student_id):
    """Every requirement, and the first course the student passed that meets it, or None if nothing does yet."""
    student = session.get(Student, student_id)
    passed = [enrollment.section.course for enrollment in student.enrollments
              if enrollment.status == "completed" and enrollment.grade != "F"]
    report = {}
    for requirement in session.scalars(select(Requirement).order_by(Requirement.name)):
        meeting = [course.code for course in passed if requirement in course.requirements]
        report[requirement.name] = meeting[0] if meeting else None
    return student, report



with SessionLocal() as session:
    for student_id in (1, 3):
        student, report = audit(session, student_id)
        print(student)
        for name, code in report.items():
            print(f"    {name:<23} {code or 'still needed'}")


Student('Ana Reyes', 'Biology')
    Humanities              HIS-110
    Lab Science             CHE-110
    Quantitative Reasoning  CSC-101
    Social Science          PSY-101
    Writing                 HIS-110
Student('Chloe Martin', 'Mathematics')
    Humanities              still needed
    Lab Science             CHE-110
    Quantitative Reasoning  still needed
    Social Science          PSY-101
    Writing                 still needed


Ana Reyes started in Fall 2024, and the courses she has passed already meet all five requirements.
Chloe Martin, with three courses behind her and a failed Data Structures that counts for nothing, has
met two. A course counts only once it is completed, so Spring 2026's courses, still under way, meet
nothing yet. The grade came from the association object and the requirements from the `secondary`
relationship, the two kinds of many to many working together.

### Where each part came from

| In `audit` | What it relies on | The section that showed it |
|---|---|---|
| `student.enrollments` with `enrollment.grade` | the association object, which reaches the grade | When the pair has data of its own: the association object, and viewonly |
| `enrollment.section.course` | many to one relationships, from the pair to its sides | When the pair has data of its own: the association object, and viewonly |
| `course.requirements` | `secondary`, the plain many to many | An association table, and secondary on both sides |
| `requirement in course.requirements` | a relationship that is a list, compared by identity | Linking and unlinking, from either side |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/12-many-to-many-solutions.ipynb).

**1.** List every course that meets Quantitative Reasoning, through `requirement.courses`.


In [8]:
# your code here


**2.** Make Composition, `ENG-105`, meet Humanities as well, by appending to the course's side, and
show the pair from the requirement's side.


In [9]:
# your code here


**3.** With a query joined through `Course.requirements`, list the courses that meet no requirement
at all. An outer join, `.outerjoin()`, keeps them.


In [10]:
# your code here


**4.** Through the read-only `Student.sections`, print the sections of Ben Okafor, student 2, and
then, through his enrollments, the grades of the ones he finished.


In [11]:
# your code here


**5.** Show that appending to the read-only `Student.sections` writes nothing: append a section,
flush, and count the student's enrollments before and after.


In [12]:
# your code here


**6.** Run `audit` for every student who started in Fall 2025, and print how many requirements each
one still needs.


In [13]:
# your code here


## Common errors

### sqlalchemy.exc.ArgumentError: secondary argument <class '__main__.Membership'> passed to to relationship() Club.members must be a Table object or other FROM clause; can't send a mapped class directly as rows in 'secondary' are persisted independently of a class that is mapped to that same table.


In [14]:
class ClubBase(DeclarativeBase):
    pass


class Member(ClubBase):
    __tablename__ = "members"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str]


class Membership(ClubBase):
    __tablename__ = "memberships"

    club_id: Mapped[int] = mapped_column(ForeignKey("clubs.id"), primary_key=True)
    member_id: Mapped[int] = mapped_column(ForeignKey("members.id"), primary_key=True)
    role: Mapped[str]


class Club(ClubBase):
    __tablename__ = "clubs"

    id: Mapped[int] = mapped_column(primary_key=True)
    members: Mapped[list["Member"]] = relationship(secondary=Membership)


Club()


ArgumentError: secondary argument <class '__main__.Membership'> passed to to relationship() Club.members must be a Table object or other FROM clause; can't send a mapped class directly as rows in 'secondary' are persisted independently of a class that is mapped to that same table.

`secondary` writes the association table's rows itself, and `Membership` is a class that would write
the same rows as objects, so SQLAlchemy refuses to hand the one to the other. The doubled "to to" is
in the library's own message. A membership has a role of its own, so it wants to be an association
object, reached through relationships, the way enrollments are. For a read-only shortcut, pass the
table and say so:


In [15]:
class ClubBase(DeclarativeBase):
    pass


class Member(ClubBase):
    __tablename__ = "members"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str]


class Membership(ClubBase):
    __tablename__ = "memberships"

    club_id: Mapped[int] = mapped_column(ForeignKey("clubs.id"), primary_key=True)
    member_id: Mapped[int] = mapped_column(ForeignKey("members.id"), primary_key=True)
    role: Mapped[str]


class Club(ClubBase):
    __tablename__ = "clubs"

    id: Mapped[int] = mapped_column(primary_key=True)
    members: Mapped[list["Member"]] = relationship(secondary=Membership.__table__, viewonly=True)


print(Club.members.property.direction.name, "through", Club.members.property.secondary.name)


MANYTOMANY through memberships


### sqlalchemy.exc.IntegrityError: (sqlite3.IntegrityError) UNIQUE constraint failed: course_requirements.course_id, course_requirements.requirement_id


In [16]:
with SessionLocal.begin() as session:
    calculus = session.scalars(select(Course).where(Course.code == "MAT-120")).one()
    quantitative = session.scalars(select(Requirement).where(Requirement.name == "Quantitative Reasoning")).one()
    calculus.requirements.append(quantitative)                       # a pair that already exists
    print("Quantitative Reasoning in the list:", calculus.requirements.count(quantitative), "times")


Quantitative Reasoning in the list: 2 times


IntegrityError: (sqlite3.IntegrityError) UNIQUE constraint failed: course_requirements.course_id, course_requirements.requirement_id
[SQL: INSERT INTO course_requirements (course_id, requirement_id) VALUES (?, ?)]
[parameters: (3, 3)]
(Background on this error at: https://sqlalche.me/e/20/gkpj)

The list accepted the requirement a second time, since a Python list holds anything twice, and the
flush tried to insert the pair again, which the association table's primary key refused. Look before
appending, or declare the collection as a set, `Mapped[set["Requirement"]]`, which cannot hold one
object twice and so never sends the second pair:


In [17]:
with SessionLocal.begin() as session:
    calculus = session.scalars(select(Course).where(Course.code == "MAT-120")).one()
    quantitative = session.scalars(select(Requirement).where(Requirement.name == "Quantitative Reasoning")).one()
    if quantitative not in calculus.requirements:
        calculus.requirements.append(quantitative)
    print("Quantitative Reasoning in the list:", calculus.requirements.count(quantitative), "time")


Quantitative Reasoning in the list: 1 time


### sqlalchemy.exc.SAWarning: relationship 'Student.courses_taken' will copy column students.id to column enrollments.student_id, which conflicts with relationship(s): 'Enrollment.student' (copies students.id to enrollments.student_id), 'Student.enrollments' (copies students.id to enrollments.student_id).


In [18]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    Student.courses_taken = relationship(Section, secondary="enrollments", order_by=Section.id)    # not viewonly
    with SessionLocal() as session:
        print(session.get(Student, 3).courses_taken)
for warning in caught:
    print(type(warning.message).__name__ + ":", str(warning.message).split(" If this is")[0])


[Section(22), Section(26), Section(29), Section(33), Section(37), Section(40)]


The new relationship could write to `enrollments`, and so can `Enrollment` and its two relationships,
so SQLAlchemy warned, once for each foreign key, that two mappings would write the same columns. It
still worked for reading, which is what makes the warning easy to dismiss, and writing through both
could produce rows neither expects. `viewonly=True`, as `Student.sections` has, says which mapping
writes: the association object.

### No error, and no way to record a grade: secondary over a table with columns of its own


In [19]:
class RosterBase(DeclarativeBase):
    pass


roster = Table(                                                  # enrollments, as secondary sees it
    "enrollments", RosterBase.metadata,
    Column("student_id", ForeignKey("students.id"), primary_key=True),
    Column("section_id", ForeignKey("sections.id"), primary_key=True),
)


class RosterSection(RosterBase):
    __tablename__ = "sections"

    id: Mapped[int] = mapped_column(primary_key=True)


class RosterStudent(RosterBase):
    __tablename__ = "students"

    id: Mapped[int] = mapped_column(primary_key=True)
    sections: Mapped[list["RosterSection"]] = relationship(secondary=roster)


with SessionLocal.begin() as session:
    ben = session.get(RosterStudent, 2)
    ben.sections.append(session.get(RosterSection, 40))          # Ben Okafor enrolls in Statistics

with engine.connect() as conn:
    print(conn.execute(text("SELECT student_id, section_id, status, grade FROM enrollments "
                            "WHERE student_id = 2 AND section_id = 40")).one())


(2, 40, 'enrolled', None)


The pair went in, and the rest of the row came from the table's defaults: `enrolled` from the
status's server default, and `NULL` for the grade. Nothing raised, and nothing in
`RosterStudent.sections` can ever read that status or record that grade, because `secondary` knows
only the two foreign keys. When a pair has data of its own, map the pair as a class, as `Enrollment`
is, and write through it:


In [20]:
with SessionLocal.begin() as session:
    enrollment = session.get(Enrollment, (2, 40))
    enrollment.grade = "B+"
    enrollment.status = "completed"
    print(enrollment.student, enrollment.section, enrollment.status, enrollment.grade)


Student('Ben Okafor', 'Computer Science') Section(40) completed B+


Last, the engine lets go of the file, and this cell removes the scratch folder, with the database in
it:


In [21]:
engine.dispose()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- A many to many relationship goes through an association table of foreign key pairs, a `Table`
  passed as `secondary`, with a list on each side linked by `back_populates`.
- Appending to either side inserts a pair and removing deletes one, leaving both objects in place;
  the table's primary key refuses the same pair twice.
- `secondary` knows only the two foreign keys, so a pair with data of its own is mapped as an
  association object, a class with a relationship to each side.
- A `secondary` relationship beside an association object is `viewonly=True`, so that only the class
  writes the table.
- A many to many relationship is a join in a query as well, through the association table.


## What is next

The **Loading Strategies** notebook counts the queries that all this navigation sends: the N plus 1
queries you did not know you wrote, `selectinload` against `joinedload`, and `unique`.


---

&#8592; **Previous:** [Relationships](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/11-relationships.ipynb)  &nbsp;·&nbsp;  [SQLAlchemy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Loading Strategies](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/13-loading-strategies.ipynb) &#8594;
